# AIODOO Colab — Train

Thin orchestration only. All training logic lives in **aiodoo-training**.

Set `AIODOO_COLAB_ROOT` to the cloned `aiodoo-colab` repository path, then run cells in order.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# Notebook is responsible for path setup (no pip install of aiodoo-colab).
COLAB_ROOT = Path(os.environ.get("AIODOO_COLAB_ROOT", ".")).resolve()
COLAB_ROOT = Path("/content/aiodoo-colab").resolve()
PYTHON_DIR = COLAB_ROOT / "python"
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))

print("aiodoo-colab root:", COLAB_ROOT)



In [ ]:
from colab_logging import configure_logging, get_logger
from config import load_config
from experiments import ExperimentStore
from models import ModelStore
from repository import TrainingRepository
from trainer import build_training_context, run_training, summarize_result
from workspace import prepare_workspace

configure_logging()
logger = get_logger()
logger.info("Modules imported")

In [ ]:
# Mount Drive + prepare AIODOO workspace layout
config = load_config()
workspace = prepare_workspace(config)
print("Workspace:", workspace.root)

In [ ]:
# Ensure frozen aiodoo-training checkout
repo = TrainingRepository.from_workspace(workspace, config)
if repo.exists():
    repo.update()
else:
    repo.clone()
repo.verify()
print("Training repository:", repo.path)

In [ ]:
# Load training config (read-only). Change TRAINING_ID as needed.
TRAINING_ID = "coding"

experiments = ExperimentStore(workspace=workspace)
experiment = experiments.load(TRAINING_ID)
print("Training:", experiment.training_id)
print("Model id:", experiment.model_id)
print("Dataset version:", experiment.dataset_version)

In [ ]:
# Ensure Hugging Face base model is available under workspace.model_cache
# (Colab local SSD: /content/aiodoo-model-cache — not Google Drive).
assert isinstance(experiment.model_id, str) and experiment.model_id.strip()
model_store = ModelStore(workspace=workspace, model_id=experiment.model_id)
model_path = model_store.ensure()
print("Model path:", model_path)

In [ ]:
# Build context and invoke aiodoo-training public entrypoint (train.py)
context = build_training_context(workspace, experiment, model_path=model_path)
result = run_training(context)
print(summarize_result(result))